In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from IPython.display import display
from maite_datasets.image_classification import MNIST

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.quality import Duplicates
from dataeval.selection import Indices, Select

set_max_processes(4)

In [3]:
# Load in the mnist dataset
testing_dataset = MNIST(root="./data/", image_set="test", download=True)

# Get the labels
labels = Metadata(testing_dataset).class_labels

In [4]:
# Creating some indices to duplicate
print("Exact duplicates")
duplicates = {}
for i in [1, 2, 5, 9]:
    matching_indices = np.where(labels == i)[0]
    print(f"\t{i} - ({matching_indices[23]}, {matching_indices[78]})")
    duplicates[int(matching_indices[78])] = int(matching_indices[23])

Exact duplicates
	1 - (180, 663)
	2 - (249, 728)
	5 - (219, 866)
	9 - (212, 773)


In [5]:
# Create a subset with the identified duplicate indices swapped
indices_with_duplicates = [duplicates.get(i, i) for i in range(len(testing_dataset))]
duplicates_ds = Select(testing_dataset, Indices(indices_with_duplicates))

In [6]:
# Initialize the Duplicates class to begin to identify duplicate images.
identifyDuplicates = Duplicates()

# Evaluate the data
results = identifyDuplicates.evaluate(duplicates_ds)

In [7]:
display(results)

shape: (133, 5)
┌──────────┬───────┬──────────┬──────────────┬────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods    │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---        │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]  │
╞══════════╪═══════╪══════════╪══════════════╪════════════╡
│ 0        ┆ item  ┆ exact    ┆ [180, 663]   ┆ ["xxhash"] │
│ 1        ┆ item  ┆ exact    ┆ [212, 773]   ┆ ["xxhash"] │
│ 2        ┆ item  ┆ exact    ┆ [219, 866]   ┆ ["xxhash"] │
│ 3        ┆ item  ┆ exact    ┆ [249, 728]   ┆ ["xxhash"] │
│ 4        ┆ item  ┆ near     ┆ [14, 8575]   ┆ ["dhash"]  │
│ …        ┆ …     ┆ …        ┆ …            ┆ …          │
│ 128      ┆ item  ┆ near     ┆ [8938, 9969] ┆ ["dhash"]  │
│ 129      ┆ item  ┆ near     ┆ [8979, 9222] ┆ ["dhash"]  │
│ 130      ┆ item  ┆ near     ┆ [8986, 9725] ┆ ["dhash"]  │
│ 131      ┆ item  ┆ near     ┆ [9689, 9819] ┆ ["dhash"]  │
│ 132      ┆ item  ┆ near     ┆ [9772, 9796] ┆ ["phash"]  │
└──────────┴───────┴────

In [8]:
### TEST ASSERTION CELL ###
assert results.exact
assert len(results.exact) == len(duplicates)
for k, v in duplicates.items():
    assert [v, k] in results.exact